# Sales channel by age group: row percentages

**Correspondence Analysis** · a worked example from the StatNex teaching guide

**The story.** A retailer published, for each age group, the share of purchases made in store, on the website, in the mobile app and via social media. Only percentages are given, not counts.

**The question.** Which channels belong with which age groups?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/correspondence_analysis__row_percent.csv` (4 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Age group,In store,Website,Mobile app,Social media,Total
20s,0.12,0.18,0.45,0.25,1.0
30s,0.18,0.27,0.4,0.15,1.0
40s,0.3,0.35,0.27,0.08,1.0
50s,0.45,0.33,0.17,0.05,1.0

These cells save `totals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# A table of row percentages: each row is one group's distribution over the columns, so
# every row sums to the same total (1 or 100%).
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["In store", "Website", "Mobile app", "Social media"]
table = data.set_index("Age group")[columns].apply(pd.to_numeric, errors="coerce")
# A row or column named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) is a margin, not data,
# so it is left out, as the app does. Spaces, underscores, case and full-width letters
# are ignored when names are compared.
from totals import is_total

table = table.loc[[not is_total(row) for row in table.index],
                  [not is_total(column) for column in table.columns]]
# A row with a blank cell is left out.
table = table.dropna()
counts = table.to_numpy(float)
n_rows, n_columns = counts.shape
row_labels, column_labels = list(table.index), list(table.columns)

## The table scaled to proportions

In [ ]:
# Each row already sums to the same total (the app refuses rows whose totals differ),
# so dividing by the grand total gives every row the same mass, 1/R for R rows: a
# profile table no longer knows how many cases each row stood for. This is the mass
# R's ca::ca and Stata's camat give a profile table.
assert np.allclose(counts.sum(axis=1), counts.sum(axis=1)[0]), "every row must sum to the same total"
P = counts / counts.sum()
row_weight = P.sum(axis=1)
row_margin, column_margin = P.sum(axis=1), P.sum(axis=0)
row_mass, column_mass = row_margin / P.sum(), column_margin / P.sum()
column_weight = column_mass

## Cross tabulation

In [ ]:
# The margin row is each column's share across the rows (its mass).
grid = np.column_stack([counts, counts.sum(axis=1)])
grid = np.vstack([grid, [*column_mass, 1.0]])
crosstab = pd.DataFrame(grid, index=[*row_labels, "Overall"], columns=[*column_labels, "Overall"])
print(crosstab)

## Decomposition

In [ ]:
# The standardised residuals from independence, (p_ij - r_i c_j) / sqrt(r_i c_j), are
# split by a singular value decomposition. A table with R rows and C columns has
# min(R, C) - 1 axes; an axis with no inertia at all is dropped.
expected = np.outer(row_margin, column_margin) / P.sum()
residual = (P - expected) / np.sqrt(expected)
U, singular, Vt = np.linalg.svd(residual, full_matrices=False)
keep = min(n_rows, n_columns) - 1
U, singular, V = U[:, :keep], singular[:keep], Vt[:keep].T
live = singular > 1e-12 * max(1.0, singular[0])
U, singular, V = U[:, live], singular[live], V[:, live]
# An eigenvector and its negative describe the same axis, so its direction is a
# convention. The app's: flip each axis so its largest row-side component (in absolute
# value) is positive. Another program may draw the map mirrored, and when two
# components tie for the largest (two rows of equal weight, say) rounding decides.
for k in range(len(singular)):
    if U[np.argmax(np.abs(U[:, k])), k] < 0:
        U[:, k], V[:, k] = -U[:, k], -V[:, k]
eigen = singular**2
share = eigen / eigen.sum()
axes = [f"Axis {k + 1}" for k in range(len(singular))]

## Eigenvalues

In [ ]:
# The eigenvalue (the singular value squared) is the inertia the axis carries; its share
# of the total decides how many axes are worth reading.
eigenvalues = pd.DataFrame({"singular value": singular, "eigenvalue": eigen, "share": share,
                            "cumulative share": share.cumsum()}, index=axes)
print()
print(eigenvalues)

## Row and column summaries

In [ ]:
# For each level and each axis:
#   eigenvector                    the singular vector itself
#   score                          eigenvector / sqrt(weight) * sqrt(singular value): the
#                                  symmetric scaling that puts rows and columns on one map
#   contribution to the axis       eigenvector^2: how much of the axis the level makes
#   contribution within the level  the share of the level's inertia the axis explains
# Mass is the level's share of the table; inertia its share of the total inertia.
def summary(labels, vectors, scores, mass):
    parts = vectors**2 * eigen
    level_totals = parts.sum(axis=1, keepdims=True)
    # A level with no inertia (a profile equal to the average one, such as a Total row)
    # gets 0 rather than 0 / 0. Its inertia is zero only up to rounding, so "zero" means
    # below 1e-12 of the total.
    zero = level_totals <= 1e-12 * eigen.sum()
    within_level = np.divide(parts, level_totals, out=np.zeros_like(parts), where=~zero)
    groups = {"Eigenvector": vectors, "Score": scores, "Contribution to the axis": vectors**2,
              "Contribution within the level": within_level}
    out = pd.DataFrame(np.hstack(list(groups.values())), index=labels,
                       columns=[f"{group} {axis}" for group in groups for axis in axes])
    out.insert(0, "Inertia", parts.sum(axis=1) / eigen.sum())
    out.insert(0, "Mass", mass)
    return out


row_scores = U / np.sqrt(row_weight)[:, None] * np.sqrt(singular)
column_scores = V / np.sqrt(column_weight)[:, None] * np.sqrt(singular)
row_summary = summary(row_labels, U, row_scores, row_mass)
column_summary = summary(column_labels, V, column_scores, column_mass)
print()
print(row_summary.round(4).T)
print()
print(column_summary.round(4).T)

## Scores

In [ ]:
# The coordinates of the map: rows and columns on the same axes. A row and a column that
# lie in the same direction from the centre occur together more often than independence
# would predict; the exact distance between a row point and a column point means little.
scores = pd.DataFrame(np.vstack([row_scores, column_scores]), index=[*row_labels, *column_labels],
                      columns=axes)
print()
print(scores.round(4))

## What the result says

Here the table holds row percentages, not counts: for each age group, the share of purchases in each channel. Correspondence analysis turns it into a map: rows and columns that go together get similar scores.

Axis 1 alone has a share of 95.1%, so one line shows almost the whole pattern. The age groups lie along it in order: 20s at 0.82, 30s at 0.32, 40s at -0.33 and 50s at -0.81. Social media (0.91) and Mobile app (0.54) are on the young side. In store (-0.76) is on the older side, and Website (-0.34) is in between.

So younger buyers lean towards the app and social media, older buyers towards the store. Every age group counts equally here, so the map shows preferences, not where most sales come from.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.